# Fishora AI Fish Identification
## DINOv3 ViT-L/16 Frozen Backbone + Head Only

Simplified **hackathon/MVP** notebook using `timm` and:

```text
vit_large_patch16_dinov3.lvd1689m
```

Only the final classification head is trained. The DINOv3 Large backbone stays fully frozen.

### Pipeline

```text
Fish image
  ↓
Frozen DINOv3 ViT-L/16
  ↓
Trainable classification head
  ↓
Fishora classes
  ↓
Temperature scaling
  ↓
Confidence threshold
  ↓
Human verification when confidence is low
```

The best head is selected using **validation Macro F1**. The test split is evaluated only after model selection.

## Kaggle setup

1. Upload the processed `fishora_dataset` directory as a Kaggle Dataset.
2. Enable **GPU T4 ×2**.
3. Enable Internet for the initial pretrained-weight download.
4. If required, add a Kaggle secret named `HF_TOKEN`.
5. Run the notebook top-to-bottom.

Expected structure:

```text
fishora_dataset/
├── cleaned/
├── metadata/
├── reports/
└── splits/
    ├── train.csv
    ├── val.csv
    └── test.csv
```

In [ ]:
!pip -q install "timm>=1.0.20" "huggingface_hub>=0.34" safetensors scikit-learn pandas pillow matplotlib tqdm

In [ ]:
import os, gc, json, math, time, random, warnings, shutil, sys
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Optional

import numpy as np
import pandas as pd
from PIL import Image, ImageOps

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

seed_everything(42)
print("PyTorch:", torch.__version__)
print("timm:", timm.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

## Configuration

The earlier frozen-head run reached very high validation performance quickly, so this notebook keeps the MVP setup simple.

If T4 memory becomes an issue, reduce `BATCH_SIZE` from 16 to 8.

In [ ]:
@dataclass
class CFG:
    DATA_ROOT: Optional[str] = None

    MODEL_NAME: str = "vit_large_patch16_dinov3.lvd1689m"
    HF_MODEL_NAME: str = "hf_hub:timm/vit_large_patch16_dinov3.lvd1689m"

    IMG_SIZE: int = 256
    BATCH_SIZE: int = 16
    NUM_WORKERS: int = 4
    USE_MULTI_GPU: bool = True

    SEED: int = 42
    EPOCHS: int = 10
    EARLY_STOP_PATIENCE: int = 3

    HEAD_LR: float = 5e-4
    WEIGHT_DECAY: float = 0.05
    LABEL_SMOOTHING: float = 0.10
    CLASS_WEIGHT_MODE: str = "sqrt_inverse"

    AMP: bool = True
    GRAD_CLIP: float = 1.0

    TARGET_ACCEPTED_ACCURACY: float = 0.95
    MIN_THRESHOLD_COVERAGE: float = 0.20

    OUTPUT_DIR: str = "/kaggle/working/fishora_dinov3_large_frozen"

cfg = CFG()
assert cfg.IMG_SIZE % 16 == 0
seed_everything(cfg.SEED)
OUT = Path(cfg.OUTPUT_DIR)
OUT.mkdir(parents=True, exist_ok=True)
print(json.dumps(asdict(cfg), indent=2))

## Locate dataset and preserve prepared splits

This notebook **does not create a new random split**. It uses the split CSVs produced during dataset preprocessing.

In [ ]:
def discover_root():
    if cfg.DATA_ROOT:
        root = Path(cfg.DATA_ROOT)
        if not root.exists():
            raise FileNotFoundError(root)
        return root

    candidates = []
    for p in Path("/kaggle/input").rglob("*"):
        if p.is_dir() and (p / "cleaned").is_dir() and (p / "splits").is_dir():
            candidates.append(p)

    if not candidates:
        raise FileNotFoundError("Could not find a directory containing cleaned/ and splits/. Set cfg.DATA_ROOT manually.")

    return sorted(candidates, key=lambda x: len(str(x)))[0]

ROOT = discover_root()
CLEANED = ROOT / "cleaned"
SPLITS = ROOT / "splits"
print("Dataset root:", ROOT)

In [ ]:
PATH_COLUMNS = [
    "resolved_path", "clean_path", "image_path", "filepath", "file_path",
    "path", "relative_path", "image", "filename", "file"
]
LABEL_COLUMNS = [
    "resolved_label", "normalized_label", "label", "class", "class_name",
    "target", "species", "folder"
]

def infer_col(df, candidates):
    mapping = {str(c).lower(): c for c in df.columns}
    for name in candidates:
        if name.lower() in mapping:
            return mapping[name.lower()]
    raise ValueError(f"Could not infer column. Available: {list(df.columns)}")

def resolve_image_path(value, csv_path):
    p = Path(str(value))
    candidates = [p] if p.is_absolute() else [ROOT / p, CLEANED / p, csv_path.parent / p]
    if not p.is_absolute() and len(p.parts) >= 2:
        candidates.append(CLEANED / Path(*p.parts[-2:]))
    for candidate in candidates:
        if candidate.exists():
            return candidate
    if not p.is_absolute() and len(p.parts) == 1:
        matches = list(CLEANED.rglob(p.name))
        if len(matches) == 1:
            return matches[0]
    raise FileNotFoundError(f"Could not resolve image path: {value}")

def load_split(kind):
    choices = {
        "train": ["train.csv"],
        "val": ["val.csv", "valid.csv", "validation.csv"],
        "test": ["test.csv"],
    }[kind]
    csv_path = next((SPLITS / x for x in choices if (SPLITS / x).exists()), None)
    if csv_path is None:
        raise FileNotFoundError(f"Missing {kind} split CSV in {SPLITS}")
    df = pd.read_csv(csv_path)
    path_col = infer_col(df, PATH_COLUMNS)
    label_col = infer_col(df, LABEL_COLUMNS)
    df = df.copy()
    df["resolved_path"] = [str(resolve_image_path(v, csv_path)) for v in df[path_col]]
    df["resolved_label"] = df[label_col].astype(str)
    return df

train_df = load_split("train")
val_df = load_split("val")
test_df = load_split("test")

CLASSES = sorted(train_df.resolved_label.unique().tolist())
C2I = {c: i for i, c in enumerate(CLASSES)}
I2C = {i: c for c, i in C2I.items()}
NUM_CLASSES = len(CLASSES)

assert not (set(train_df.resolved_path) & set(val_df.resolved_path))
assert not (set(train_df.resolved_path) & set(test_df.resolved_path))
assert not (set(val_df.resolved_path) & set(test_df.resolved_path))

print("Classes:", NUM_CLASSES, CLASSES)
display(pd.DataFrame({
    "train": train_df.resolved_label.value_counts(),
    "val": val_df.resolved_label.value_counts(),
    "test": test_df.resolved_label.value_counts(),
}).fillna(0).astype(int).reindex(CLASSES))

In [ ]:
def show_samples(df, n=12):
    sample = df.sample(min(n, len(df)), random_state=cfg.SEED)
    cols = 4
    rows = math.ceil(len(sample) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(14, 3.5 * rows))
    axes = np.array(axes).reshape(-1)
    for ax in axes:
        ax.axis("off")
    for ax, (_, row) in zip(axes, sample.iterrows()):
        ax.imshow(Image.open(row.resolved_path).convert("RGB"))
        ax.set_title(row.resolved_label)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

show_samples(train_df)

## Load DINOv3 Large with timm

In [ ]:
def get_hf_token():
    if os.getenv("HF_TOKEN"):
        return os.getenv("HF_TOKEN")
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return None

HF_TOKEN = get_hf_token()
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

def create_model(pretrained=True):
    errors = []
    for model_name in [cfg.MODEL_NAME, cfg.HF_MODEL_NAME]:
        try:
            model = timm.create_model(
                model_name,
                pretrained=pretrained,
                num_classes=NUM_CLASSES,
            )
            try:
                model.reset_classifier(num_classes=NUM_CLASSES, global_pool="token")
            except Exception:
                pass
            return model
        except Exception as e:
            errors.append((model_name, repr(e)))
    raise RuntimeError("Failed to create DINOv3 Large:\n" + "\n".join(f"{n}: {e}" for n, e in errors))

probe = create_model(pretrained=False)
data_cfg = timm.data.resolve_data_config(probe.pretrained_cfg)
MEAN = tuple(data_cfg["mean"])
STD = tuple(data_cfg["std"])
FILL_RGB = tuple(int(round(x * 255)) for x in MEAN)
print("Parameters:", f"{sum(p.numel() for p in probe.parameters()) / 1e6:.1f}M")
print("Mean:", MEAN)
print("Std:", STD)
del probe
gc.collect()

## Fish-safe preprocessing

Aspect ratio is preserved and the image is padded. We avoid aggressive crops so the head, tail, fins, body shape, and markings remain visible.

In [ ]:
class ResizePad:
    def __init__(self, size, fill):
        self.size = int(size)
        self.fill = tuple(fill)

    def __call__(self, image):
        image = ImageOps.exif_transpose(image).convert("RGB")
        w, h = image.size
        scale = min(self.size / w, self.size / h)
        nw = max(1, int(round(w * scale)))
        nh = max(1, int(round(h * scale)))
        image = image.resize((nw, nh), Image.Resampling.BICUBIC)
        canvas = Image.new("RGB", (self.size, self.size), self.fill)
        canvas.paste(image, ((self.size - nw) // 2, (self.size - nh) // 2))
        return canvas

train_tf = transforms.Compose([
    ResizePad(cfg.IMG_SIZE, FILL_RGB),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomAffine(degrees=8, translate=(0.04, 0.04), scale=(0.94, 1.04), fill=FILL_RGB),
    transforms.ColorJitter(brightness=0.12, contrast=0.12, saturation=0.10, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

eval_tf = transforms.Compose([
    ResizePad(cfg.IMG_SIZE, FILL_RGB),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

In [ ]:
class FishDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row.resolved_path).convert("RGB")
        image = self.transform(image)
        target = C2I[row.resolved_label]
        return image, target, row.resolved_path

loader_kwargs = dict(
    batch_size=cfg.BATCH_SIZE,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(cfg.NUM_WORKERS > 0),
)

train_loader = DataLoader(FishDataset(train_df, train_tf), shuffle=True, **loader_kwargs)
val_loader = DataLoader(FishDataset(val_df, eval_tf), shuffle=False, **loader_kwargs)
test_loader = DataLoader(FishDataset(test_df, eval_tf), shuffle=False, **loader_kwargs)

x, y, p = next(iter(train_loader))
print("Batch:", x.shape, y.shape)

## Freeze DINOv3 completely and train only the classifier head

In [ ]:
model = create_model(pretrained=True)

for p in model.parameters():
    p.requires_grad = False

classifier = model.get_classifier()
for p in classifier.parameters():
    p.requires_grad = True

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.5f}%)")

In [ ]:
counts = train_df.resolved_label.value_counts().reindex(CLASSES).values.astype(float)
if cfg.CLASS_WEIGHT_MODE == "none":
    weights = np.ones_like(counts)
elif cfg.CLASS_WEIGHT_MODE == "inverse":
    weights = 1.0 / counts
elif cfg.CLASS_WEIGHT_MODE == "sqrt_inverse":
    weights = 1.0 / np.sqrt(counts)
else:
    raise ValueError(cfg.CLASS_WEIGHT_MODE)
weights = weights / weights.mean()
CLASS_WEIGHTS = torch.tensor(weights, dtype=torch.float32)
display(pd.DataFrame({"class": CLASSES, "train_count": counts.astype(int), "loss_weight": weights}))

## Training utilities

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = bool(cfg.AMP and torch.cuda.is_available())

if cfg.USE_MULTI_GPU and torch.cuda.is_available() and torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
    print(f"Using nn.DataParallel on {torch.cuda.device_count()} GPUs.")

model = model.to(DEVICE)

def unwrap(m):
    return m.module if isinstance(m, nn.DataParallel) else m

criterion = nn.CrossEntropyLoss(
    weight=CLASS_WEIGHTS.to(DEVICE),
    label_smoothing=cfg.LABEL_SMOOTHING,
)

optimizer = torch.optim.AdamW(
    unwrap(model).get_classifier().parameters(),
    lr=cfg.HEAD_LR,
    weight_decay=cfg.WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.EPOCHS)
scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)

def compute_metrics(y_true, logits):
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    pred = probs.argmax(axis=1)
    top1 = accuracy_score(y_true, pred)
    macro_f1 = f1_score(y_true, pred, average="macro")
    k = min(3, probs.shape[1])
    topk = np.argpartition(-probs, kth=k-1, axis=1)[:, :k]
    top3 = np.mean([y in row for y, row in zip(y_true, topk)])
    return {"top1": float(top1), "top3": float(top3), "macro_f1": float(macro_f1)}

In [ ]:
@torch.no_grad()
def evaluate(loader):
    model.eval()
    total_loss, n_seen = 0.0, 0
    logits_all, labels_all, paths_all = [], [], []

    for x, y, paths in tqdm(loader, leave=False):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP_ENABLED):
            logits = model(x)
            loss = criterion(logits, y)
        total_loss += loss.item() * len(y)
        n_seen += len(y)
        logits_all.append(logits.float().cpu())
        labels_all.append(y.cpu())
        paths_all.extend(paths)

    logits_np = torch.cat(logits_all).numpy()
    y_true = torch.cat(labels_all).numpy()
    metrics = compute_metrics(y_true, logits_np)
    metrics["loss"] = total_loss / max(1, n_seen)
    return metrics, logits_np, y_true, paths_all

def train_one_epoch():
    model.train()
    total_loss, n_seen = 0.0, 0
    logits_all, labels_all = [], []

    for x, y, _ in tqdm(train_loader, leave=False):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP_ENABLED):
            logits = model(x)
            loss = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(unwrap(model).get_classifier().parameters(), cfg.GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * len(y)
        n_seen += len(y)
        logits_all.append(logits.detach().float().cpu())
        labels_all.append(y.detach().cpu())

    logits_np = torch.cat(logits_all).numpy()
    y_true = torch.cat(labels_all).numpy()
    metrics = compute_metrics(y_true, logits_np)
    metrics["loss"] = total_loss / max(1, n_seen)
    return metrics

## Train head only

Early stopping is based on validation Macro F1.

In [ ]:
BEST_PATH = OUT / "fishora_dinov3_large_frozen_best.pt"
history = []
best_f1 = -1.0
no_improve = 0

for epoch in range(1, cfg.EPOCHS + 1):
    start = time.time()
    train_metrics = train_one_epoch()
    val_metrics, _, _, _ = evaluate(val_loader)
    scheduler.step()
    elapsed = (time.time() - start) / 60.0

    row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_top1": train_metrics["top1"],
        "train_macro_f1": train_metrics["macro_f1"],
        "val_loss": val_metrics["loss"],
        "val_top1": val_metrics["top1"],
        "val_top3": val_metrics["top3"],
        "val_macro_f1": val_metrics["macro_f1"],
        "minutes": elapsed,
    }
    history.append(row)

    print(
        f"[frozen_head] {epoch:02d}/{cfg.EPOCHS} | "
        f"train F1={train_metrics['macro_f1']:.4f} | "
        f"val F1={val_metrics['macro_f1']:.4f} | "
        f"val top1={val_metrics['top1']:.4f} | "
        f"{elapsed:.1f} min"
    )

    if val_metrics["macro_f1"] > best_f1 + 1e-4:
        best_f1 = val_metrics["macro_f1"]
        no_improve = 0
        torch.save(unwrap(model).state_dict(), BEST_PATH)
    else:
        no_improve += 1

    if no_improve >= cfg.EARLY_STOP_PATIENCE:
        print("Early stopping.")
        break

history_df = pd.DataFrame(history)
history_df.to_csv(OUT / "training_history.csv", index=False)
display(history_df)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history_df.epoch, history_df.train_macro_f1, label="Train Macro F1")
plt.plot(history_df.epoch, history_df.val_macro_f1, label="Validation Macro F1")
plt.xlabel("Epoch")
plt.ylabel("Macro F1")
plt.title("Frozen DINOv3 Large")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## Evaluate best frozen-head model on test set

In [ ]:
unwrap(model).load_state_dict(torch.load(BEST_PATH, map_location="cpu", weights_only=True))
model = model.to(DEVICE)

val_metrics, val_logits, val_y, val_paths = evaluate(val_loader)
test_metrics, test_logits, test_y, test_paths = evaluate(test_loader)

print("Validation:")
print(json.dumps(val_metrics, indent=2))
print("\nTest:")
print(json.dumps(test_metrics, indent=2))

In [ ]:
test_probs = torch.softmax(torch.tensor(test_logits), dim=1).numpy()
test_pred = test_probs.argmax(axis=1)

report = pd.DataFrame(classification_report(
    test_y,
    test_pred,
    labels=list(range(NUM_CLASSES)),
    target_names=CLASSES,
    output_dict=True,
    zero_division=0,
)).T

display(report)
report.to_csv(OUT / "test_classification_report.csv")

cm = confusion_matrix(test_y, test_pred, labels=list(range(NUM_CLASSES)))
fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(cm)
ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(CLASSES, rotation=90); ax.set_yticklabels(CLASSES)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Fishora DINOv3 Large — Confusion Matrix")
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, cm[i, j], ha="center", va="center")
plt.tight_layout(); plt.show()

## Temperature scaling and confidence threshold

This calibrates confidence using the validation split. Low-confidence cases should go to **human verification**.

In [ ]:
class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.log_temperature = nn.Parameter(torch.zeros(1))

    @property
    def temperature(self):
        return self.log_temperature.exp().clamp(0.05, 20.0)

    def forward(self, logits):
        return logits / self.temperature

def fit_temperature(logits_np, labels_np):
    logits = torch.tensor(logits_np, dtype=torch.float32, device=DEVICE)
    labels = torch.tensor(labels_np, dtype=torch.long, device=DEVICE)
    scaler_model = TemperatureScaler().to(DEVICE)
    ce = nn.CrossEntropyLoss()
    opt = torch.optim.LBFGS([scaler_model.log_temperature], lr=0.05, max_iter=100, line_search_fn="strong_wolfe")

    def closure():
        opt.zero_grad()
        loss = ce(scaler_model(logits), labels)
        loss.backward()
        return loss

    opt.step(closure)
    return float(scaler_model.temperature.detach().cpu())

TEMPERATURE = fit_temperature(val_logits, val_y)
print("Temperature:", TEMPERATURE)

def calibrated_probs(logits):
    return torch.softmax(torch.tensor(logits, dtype=torch.float32) / TEMPERATURE, dim=1).numpy()

val_probs_cal = calibrated_probs(val_logits)
test_probs_cal = calibrated_probs(test_logits)

In [ ]:
def choose_threshold(probs, y_true):
    confidence = probs.max(axis=1)
    prediction = probs.argmax(axis=1)
    rows = []

    for threshold in np.linspace(0.0, 0.99, 200):
        accepted = confidence >= threshold
        if not accepted.any():
            continue
        rows.append({
            "threshold": float(threshold),
            "coverage": float(accepted.mean()),
            "accepted_accuracy": float((prediction[accepted] == y_true[accepted]).mean()),
        })

    table = pd.DataFrame(rows)
    valid = table[
        (table.accepted_accuracy >= cfg.TARGET_ACCEPTED_ACCURACY)
        & (table.coverage >= cfg.MIN_THRESHOLD_COVERAGE)
    ]

    if len(valid):
        selected = valid.sort_values("threshold").iloc[0]
    else:
        selected = table.sort_values(["accepted_accuracy", "coverage"], ascending=False).iloc[0]

    return float(selected.threshold), table

ABSTAIN_THRESHOLD, threshold_table = choose_threshold(val_probs_cal, val_y)
print("Abstain threshold:", ABSTAIN_THRESHOLD)

## Export backend-ready model

Output:

```text
export/
├── model_state_dict.pt
├── inference_config.json
└── inference.py
```

In [ ]:
EXPORT = OUT / "export"
EXPORT.mkdir(parents=True, exist_ok=True)
core = unwrap(model)

torch.save(core.state_dict(), EXPORT / "model_state_dict.pt")

inference_config = {
    "library": "timm",
    "model_name": cfg.MODEL_NAME,
    "num_classes": NUM_CLASSES,
    "classes": CLASSES,
    "class_to_idx": C2I,
    "img_size": cfg.IMG_SIZE,
    "mean": list(MEAN),
    "std": list(STD),
    "fill_rgb": list(FILL_RGB),
    "temperature": TEMPERATURE,
    "abstain_threshold": ABSTAIN_THRESHOLD,
    "training_strategy": "frozen_backbone_head_only",
    "validation_metrics": val_metrics,
    "test_metrics": test_metrics,
}

with open(EXPORT / "inference_config.json", "w", encoding="utf-8") as f:
    json.dump(inference_config, f, indent=2, ensure_ascii=False)

print(json.dumps(inference_config, indent=2))

In [ ]:
INFERENCE_PY = r"""import json
from pathlib import Path
import torch
import timm
from PIL import Image, ImageOps
from torchvision import transforms

class ResizePad:
    def __init__(self, size, fill):
        self.size = int(size)
        self.fill = tuple(fill)

    def __call__(self, image):
        image = ImageOps.exif_transpose(image).convert("RGB")
        w, h = image.size
        scale = min(self.size / w, self.size / h)
        nw = max(1, int(round(w * scale)))
        nh = max(1, int(round(h * scale)))
        image = image.resize((nw, nh), Image.Resampling.BICUBIC)
        canvas = Image.new("RGB", (self.size, self.size), self.fill)
        canvas.paste(image, ((self.size - nw) // 2, (self.size - nh) // 2))
        return canvas

class FishoraClassifier:
    def __init__(self, export_dir, device=None):
        export_dir = Path(export_dir)
        with open(export_dir / "inference_config.json", "r", encoding="utf-8") as f:
            self.cfg = json.load(f)

        self.device = torch.device(device if device else ("cuda" if torch.cuda.is_available() else "cpu"))

        self.model = timm.create_model(
            self.cfg["model_name"],
            pretrained=False,
            num_classes=self.cfg["num_classes"],
        )
        try:
            self.model.reset_classifier(num_classes=self.cfg["num_classes"], global_pool="token")
        except Exception:
            pass

        state = torch.load(export_dir / "model_state_dict.pt", map_location="cpu", weights_only=True)
        self.model.load_state_dict(state)
        self.model = self.model.to(self.device).eval()

        self.transform = transforms.Compose([
            ResizePad(self.cfg["img_size"], self.cfg["fill_rgb"]),
            transforms.ToTensor(),
            transforms.Normalize(self.cfg["mean"], self.cfg["std"]),
        ])

    @torch.inference_mode()
    def predict(self, image, top_k=3):
        if isinstance(image, (str, Path)):
            image = Image.open(image)

        x = self.transform(image.convert("RGB")).unsqueeze(0).to(self.device)
        logits = self.model(x)
        probs = torch.softmax(logits.float() / float(self.cfg["temperature"]), dim=1)[0]

        k = min(top_k, len(self.cfg["classes"]))
        values, indices = probs.topk(k)
        candidates = [
            {"label": self.cfg["classes"][int(idx)], "confidence": float(value)}
            for value, idx in zip(values.cpu(), indices.cpu())
        ]

        best = candidates[0]
        threshold = float(self.cfg["abstain_threshold"])

        return {
            "status": (
                "confident_prediction"
                if best["confidence"] >= threshold
                else "low_confidence_human_verification_required"
            ),
            "prediction": best,
            "top_candidates": candidates,
            "threshold": threshold,
        }
"""

with open(EXPORT / "inference.py", "w", encoding="utf-8") as f:
    f.write(INFERENCE_PY)

print("Created:", EXPORT / "inference.py")

## Test exported backend wrapper

In [ ]:
sys.path.insert(0, str(EXPORT))
from inference import FishoraClassifier

backend_model = FishoraClassifier(
    EXPORT,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

sample_path = test_df.iloc[0].resolved_path
result = backend_model.predict(sample_path, top_k=3)
print(json.dumps(result, indent=2))

## Package Kaggle outputs

In [ ]:
summary = {
    "dataset_root": str(ROOT),
    "model": cfg.MODEL_NAME,
    "strategy": "frozen_backbone_head_only",
    "num_classes": NUM_CLASSES,
    "classes": CLASSES,
    "split_sizes": {
        "train": len(train_df),
        "val": len(val_df),
        "test": len(test_df),
    },
    "best_validation_macro_f1": best_f1,
    "validation_metrics": val_metrics,
    "test_metrics": test_metrics,
    "temperature": TEMPERATURE,
    "abstain_threshold": ABSTAIN_THRESHOLD,
}

with open(OUT / "final_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

archive = shutil.make_archive(
    "/kaggle/working/fishora_dinov3_large_frozen_artifacts",
    "zip",
    root_dir=OUT,
)

print("Artifact:", archive)

## MVP note

For the current hackathon, the frozen-head approach is enough if:

- test performance is strong,
- inference is stable,
- low-confidence cases fall back to human verification,
- the backend can load the exported model successfully.

If this MVP later becomes a production system, prioritize diverse real landing-point images and unsupported-species/OOD evaluation before deciding whether the DINOv3 backbone needs fine-tuning.